# 03 — Avaliação (nDCG@k) e Análise dos Resultados

Executa as buscas de todas as queries do split de teste (locale US) e calcula nDCG@k para k = 5, 10, 20 e 45
nas estratégias **BM25**, **KNN** e **Hybrid** (RRF), e depois analisa os resultados.
Este notebook substitui o antigo `scripts/04_run_evaluation.py`.

**Pré-requisitos**
- `data/processed/examples_us.parquet` (notebook 01)
- Índices criados no Elasticsearch local com `scripts/index_embeddings.py` (valide com o notebook 02b),
  com o servidor ativo. Este notebook roda na máquina local; a avaliação KNN carrega os modelos para
  codificar as queries.

Os resultados são salvos em `results/elasticsearch/` e uma execução interrompida pode ser retomada
(`RESUME = True`). Para alternar CPU/GPU, altere apenas `DEVICE` na célula de setup.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

# ── Configuração única ───────────────────────────────────────────────
DEVICE = "auto"                     # "cpu" | "cuda" | "auto"
DRIVE_PROJECT_DIR = "MyDrive/tcc"   # pasta do projeto no Google Drive (só Colab)
# ─────────────────────────────────────────────────────────────────────

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive") / DRIVE_PROJECT_DIR
    %pip install -q polars pyarrow matplotlib numpy elasticsearch~=8.15 sentence-transformers tqdm
else:
    PROJECT_ROOT = Path.cwd()

assert (PROJECT_ROOT / "src").exists(), f"Pasta 'src' não encontrada em {PROJECT_ROOT}"
sys.path.insert(0, str(PROJECT_ROOT))

from src.device import build_runtime_config, collect

RUNTIME = build_runtime_config(DEVICE)
print(f"Projeto : {PROJECT_ROOT}")
print(f"Colab   : {IN_COLAB}")
print(f"Device  : {RUNTIME.device} (batch_size={RUNTIME.batch_size}, max_seq_length={RUNTIME.max_seq_length})")

In [ ]:
import json
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from elasticsearch import Elasticsearch
from tqdm.auto import tqdm

from src.elasticsearch.searcher import Searcher, SearchStrategy
from src.models.core import unload_all_models
from src.models.registry import BM25_INDEX_NAME, get_model_config, list_models

# ── Parâmetros da avaliação ──────────────────────────────────────────
MODELS: list[str] | None = None        # None = todos; ex.: ["bge", "rexbert"]
STRATEGIES: list[str] | None = None    # None = todas; ex.: ["knn"]
LIMIT_QUERIES: int | None = None       # use um valor pequeno para testes rápidos
RESUME = True                          # reaproveita JSONs já salvos
K_VALUES = [5, 10, 20, 45]
K_MAX = max(K_VALUES)
ESCI_GAINS = {"E": 1.0, "S": 0.1, "C": 0.01, "I": 0.0}

# ── Elasticsearch ────────────────────────────────────────────────────
ES_URL = os.environ.get("ES_URL", "http://localhost:9200")
ES_USER = os.environ.get("ES_USER", "elastic")
ES_PASSWORD = os.environ.get("ES_PASSWORD")   # None = sem autenticação

EXAMPLES_PATH = PROJECT_ROOT / "data" / "processed" / "examples_us.parquet"
# Execuções parciais não devem poluir os resultados oficiais lidos pelo notebook 04.
RESULTS_DIR = PROJECT_ROOT / "results" / ("elasticsearch_debug" if LIMIT_QUERIES else "elasticsearch")
REPORT_DIR = PROJECT_ROOT / "results" / "report"
for directory in (RESULTS_DIR, REPORT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

assert EXAMPLES_PATH.exists(), f"{EXAMPLES_PATH} não encontrado. Execute o notebook 01 primeiro."
plt.rcParams["figure.dpi"] = 120

## 1. Conexão com o Elasticsearch

In [ ]:
client_kwargs: dict = {"request_timeout": 60}
if ES_PASSWORD:
    client_kwargs["basic_auth"] = (ES_USER, ES_PASSWORD)
if ES_URL.startswith("https"):
    client_kwargs["verify_certs"] = False   # certificado autoassinado do Docker
es = Elasticsearch(ES_URL, **client_kwargs)

try:
    print("Elasticsearch", es.info()["version"]["number"], "conectado em", ES_URL)
except Exception as exc:
    raise RuntimeError(
        f"Sem conexão com {ES_URL}: {exc}\nInicie o Elasticsearch (Docker) e confira ES_URL, ES_USER e ES_PASSWORD."
    ) from exc

all_aliases = [cfg.alias for cfg in list_models()]
target_aliases = MODELS or all_aliases
target_strategies = STRATEGIES or [s.value for s in SearchStrategy]

needed = [BM25_INDEX_NAME] + [get_model_config(a).es_index_name for a in target_aliases]
absent = [name for name in needed if not es.indices.exists(index=name)]
assert not absent, f"Índices ausentes: {absent}. Execute scripts/index_embeddings.py."

## 2. Dados de teste e qrels

In [ ]:
examples = pl.read_parquet(EXAMPLES_PATH)
test_df = examples.filter(pl.col("split") == "test")

if LIMIT_QUERIES:
    keep_ids = test_df["query_id"].unique(maintain_order=True).head(LIMIT_QUERIES)
    test_df = test_df.filter(pl.col("query_id").is_in(keep_ids))
    print(f"AVISO: avaliação limitada a {LIMIT_QUERIES} queries; resultados em {RESULTS_DIR.name}/")

# {query_id: {product_id: ganho ESCI}}
qrels_map: dict = {}
for query_id, product_id, label in test_df.select("query_id", "product_id", "esci_label").iter_rows():
    qrels_map.setdefault(query_id, {})[product_id] = ESCI_GAINS.get(label, 0.0)

# Queries sem nenhum produto com ganho > 0 não contribuem para o nDCG
eval_queries = [
    (query_id, text)
    for query_id, text in test_df.unique(subset="query_id", maintain_order=True).select("query_id", "query").iter_rows()
    if any(gain > 0 for gain in qrels_map[query_id].values())
]
print(f"Pares de teste: {test_df.height:,} | queries avaliáveis: {len(eval_queries):,}")

## 3. Métrica e avaliação

In [ ]:
DISCOUNTS = 1.0 / np.log2(np.arange(2, K_MAX + 2))


def dcg_at_k(gains: list[float], k: int) -> float:
    top = np.asarray(gains[:k], dtype=float)
    return float((top * DISCOUNTS[: len(top)]).sum())


def ndcg_at_k(retrieved_ids: list[str], qrels: dict[str, float], k: int) -> float:
    gained = [qrels.get(pid, 0.0) for pid in retrieved_ids[:k]]
    ideal = sorted(qrels.values(), reverse=True)[:k]
    idcg = dcg_at_k(ideal, k)
    return dcg_at_k(gained, k) / idcg if idcg > 0 else 0.0


# Verificações rápidas da métrica
assert ndcg_at_k(["a", "b"], {"a": 1.0, "b": 0.1}, 2) == 1.0
assert ndcg_at_k(["b", "a"], {"a": 1.0, "b": 0.1}, 2) < 1.0
assert ndcg_at_k(["x"], {"a": 1.0}, 5) == 0.0


def evaluate(searcher: Searcher, alias: str, strategy: str) -> dict:
    """Roda todas as queries e retorna métricas agregadas (dict vazio se a estratégia falhar)."""
    strategy_enum = SearchStrategy(strategy)

    try:  # aquece o modelo para que o carregamento não entre na latência
        searcher.search(alias, eval_queries[0][1], strategy=strategy_enum, k=K_MAX)
    except Exception:
        pass

    per_query: list[dict[str, float]] = []
    latencies: list[float] = []
    errors = 0
    for query_id, text in tqdm(eval_queries, desc=f"{alias}/{strategy}", leave=False):
        started = time.perf_counter()
        try:
            hits = searcher.search(alias, text, strategy=strategy_enum, k=K_MAX)
        except Exception as exc:
            errors += 1
            if errors == 1:
                print(f"  erro em {alias}/{strategy}: {exc}")
            if not latencies and errors >= 3:
                print("  abortando: falhas consecutivas desde o início.")
                return {}
            continue
        latencies.append((time.perf_counter() - started) * 1000)
        retrieved = [hit.product_id for hit in hits]
        per_query.append({f"ndcg@{k}": ndcg_at_k(retrieved, qrels_map[query_id], k) for k in K_VALUES})

    if not per_query:
        return {}
    means = pl.DataFrame(per_query).mean().row(0, named=True)
    return {
        "model": alias,
        "strategy": strategy,
        "locale": "us",
        "n_queries": len(per_query),
        "n_errors": errors,
        "latency_ms_mean": round(float(np.mean(latencies)), 2),
        "latency_ms_p95": round(float(np.percentile(latencies, 95)), 2),
        "metrics": {name: round(float(value), 4) for name, value in means.items()},
    }

## 4. Execução

BM25 não depende do modelo e roda uma única vez. Cada resultado é salvo em
`results/elasticsearch/<modelo>_<estratégia>.json`.

In [ ]:
searcher = Searcher(es, device=RUNTIME.device)


def result_path(alias: str, strategy: str):
    return RESULTS_DIR / ("bm25.json" if strategy == "bm25" else f"{alias}_{strategy}.json")


def run_or_load(alias: str, strategy: str) -> dict:
    path = result_path(alias, strategy)
    if RESUME and path.exists():
        print(f"{alias}/{strategy}: reaproveitando {path.name}")
        return json.loads(path.read_text(encoding="utf-8"))
    result = evaluate(searcher, alias, strategy)
    if result:
        path.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")
        print(
            f"{alias}/{strategy}: nDCG@10={result['metrics']['ndcg@10']:.4f} | "
            f"latência={result['latency_ms_mean']:.1f} ms (p95 {result['latency_ms_p95']:.1f})"
        )
    return result


all_results: list[dict] = []

if "bm25" in target_strategies:
    if result := run_or_load("bm25", "bm25"):
        all_results.append(result)

for alias in target_aliases:
    for strategy in target_strategies:
        if strategy == "bm25":
            continue
        if result := run_or_load(alias, strategy):
            all_results.append(result)
    searcher.unload_model(alias)

consolidated_path = RESULTS_DIR / "all_results.json"
consolidated_path.write_text(json.dumps(all_results, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"\n{len(all_results)} experimentos salvos em {consolidated_path}")
unload_all_models()

## 5. Carregamento dos resultados

In [ ]:
all_results = json.loads((RESULTS_DIR / "all_results.json").read_text(encoding="utf-8"))
assert all_results, "Nenhum resultado disponível."

MODEL_ORDER = ["bm25"] + [cfg.alias for cfg in list_models()]
DISPLAY = {"bm25": "BM25"} | {cfg.alias: cfg.display_name for cfg in list_models()}
palette = plt.get_cmap("tab10")
MODEL_COLORS = {model: palette(i) for i, model in enumerate(MODEL_ORDER)}
STRATEGY_STYLES = {
    "bm25": {"linestyle": "--", "marker": "s"},
    "knn": {"linestyle": "-", "marker": "o"},
    "hybrid": {"linestyle": "-.", "marker": "^"},
}

df = (
    pl.DataFrame(
        [
            {
                "model": r["model"],
                "strategy": r["strategy"],
                "n_queries": r.get("n_queries"),
                "latency_ms_mean": r.get("latency_ms_mean"),
                "latency_ms_p95": r.get("latency_ms_p95"),
                **{f"ndcg@{k}": r["metrics"].get(f"ndcg@{k}") for k in K_VALUES},
            }
            for r in all_results
        ]
    )
    .with_columns(_order=pl.col("model").replace_strict({m: i for i, m in enumerate(MODEL_ORDER)}, default=99))
    .sort("_order", "strategy")
    .drop("_order")
)


def ndcg10(model: str, strategy: str) -> float | None:
    values = df.filter((pl.col("model") == model) & (pl.col("strategy") == strategy))["ndcg@10"]
    return values[0] if len(values) else None


df

## 6. Curva nDCG@k de todos os experimentos

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
for row in df.iter_rows(named=True):
    values = [row[f"ndcg@{k}"] for k in K_VALUES]
    if any(v is None for v in values):
        continue
    ax.plot(
        K_VALUES,
        values,
        label=f"{DISPLAY.get(row['model'], row['model'])} / {row['strategy']}",
        color=MODEL_COLORS.get(row["model"], "#333333"),
        linewidth=1.8,
        markersize=6,
        **STRATEGY_STYLES.get(row["strategy"], {"linestyle": "-", "marker": "x"}),
    )
ax.set_xlabel("k")
ax.set_ylabel("nDCG@k")
ax.set_title("Curva nDCG@k — todos os modelos e estratégias")
ax.set_xticks(K_VALUES)
ax.set_ylim(bottom=0)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
ax.grid(True, linestyle="--", alpha=0.4)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(REPORT_DIR / "eval_ndcg_curve_all.png", bbox_inches="tight")
plt.show()

## 7. KNN e Hybrid por modelo (nDCG@10)

A linha tracejada é o BM25, usado como referência.

In [ ]:
embedding_models = [m for m in MODEL_ORDER[1:] if m in df["model"].to_list()]
strategies = ["knn", "hybrid"]
strategy_colors = {"knn": "#1f77b4", "hybrid": "#2ca02c"}

x = np.arange(len(embedding_models))
width = 0.38
fig, ax = plt.subplots(figsize=(11, 5))
for i, strategy in enumerate(strategies):
    values = [ndcg10(m, strategy) or 0.0 for m in embedding_models]
    bars = ax.bar(x + (i - 0.5) * width, values, width, label=strategy.upper(), color=strategy_colors[strategy], alpha=0.85)
    ax.bar_label(bars, fmt="%.3f", fontsize=7, rotation=90, padding=2)

if (bm25_value := ndcg10("bm25", "bm25")) is not None:
    ax.axhline(bm25_value, color="gray", linestyle="--", label=f"BM25 ({bm25_value:.3f})")

ax.set_xticks(x, [DISPLAY[m] for m in embedding_models])
ax.set_ylabel("nDCG@10")
ax.set_title("nDCG@10 por modelo e estratégia")
ax.set_ylim(0, df["ndcg@10"].max() * 1.2)
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(REPORT_DIR / "eval_ndcg10_grouped.png", bbox_inches="tight")
plt.show()

## 8. Ganho relativo sobre o BM25

In [ ]:
bm25_value = ndcg10("bm25", "bm25")
if bm25_value:
    print(f"BM25 nDCG@10 = {bm25_value:.4f} (referência = 0%)")
    gain_df = (
        df.filter(pl.col("strategy").is_in(["knn", "hybrid"]))
        .select("model", "strategy", "ndcg@10", gain_vs_bm25_pct=((pl.col("ndcg@10") - bm25_value) / bm25_value * 100).round(2))
        .sort("gain_vs_bm25_pct", descending=True)
    )
    gain_df
else:
    print("BM25 não encontrado nos resultados.")

## 9. Hipótese: contexto longo vs 512 tokens (KNN)

In [ ]:
max_seq = {cfg.alias: cfg.max_seq_length for cfg in list_models()}
knn_df = (
    df.filter(pl.col("strategy") == "knn")
    .with_columns(max_seq=pl.col("model").replace_strict(max_seq, default=None))
    .drop_nulls(["ndcg@10", "max_seq"])
)

fig, ax = plt.subplots(figsize=(9, 4))
colors = ["#2ca02c" if s > 512 else "#1f77b4" for s in knn_df["max_seq"]]
bars = ax.bar([DISPLAY[m] for m in knn_df["model"]], knn_df["ndcg@10"], color=colors, alpha=0.85)
ax.bar_label(bars, fmt="%.4f", fontsize=9)
ax.legend(
    handles=[
        plt.Rectangle((0, 0), 1, 1, color="#1f77b4", label="≤ 512 tokens"),
        plt.Rectangle((0, 0), 1, 1, color="#2ca02c", label="> 512 tokens"),
    ],
    title="max_seq_length",
)
ax.set_ylabel("nDCG@10 (KNN)")
ax.set_title("Impacto do contexto longo no nDCG@10")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(REPORT_DIR / "eval_context_length_impact.png", bbox_inches="tight")
plt.show()

## 10. Hipótese: BERTimbau em inglês

In [ ]:
compare_models = [m for m in ("bert", "roberta", "bertimbau") if m in df["model"].to_list()]

fig, ax = plt.subplots(figsize=(8, 4))
for strategy, style in (("knn", "-o"), ("hybrid", "--^")):
    values = [ndcg10(m, strategy) if ndcg10(m, strategy) is not None else np.nan for m in compare_models]
    ax.plot([DISPLAY[m] for m in compare_models], values, style, label=strategy.upper(), linewidth=1.8, markersize=8)
ax.set_ylabel("nDCG@10")
ax.set_title("BERT vs RoBERTa vs BERTimbau (locale = EN)")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.4)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(REPORT_DIR / "eval_bertimbau_comparison.png", bbox_inches="tight")
plt.show()

df.filter(pl.col("model").is_in(compare_models) & pl.col("strategy").is_in(["knn", "hybrid"])).select(
    "model", "strategy", *[f"ndcg@{k}" for k in K_VALUES]
).sort("strategy", "model")

## 11. Trade-off qualidade × latência

In [ ]:
lat_df = df.drop_nulls(["latency_ms_mean", "ndcg@10"])
markers = {"bm25": "s", "knn": "o", "hybrid": "^"}

fig, ax = plt.subplots(figsize=(9, 5))
for row in lat_df.iter_rows(named=True):
    ax.scatter(
        row["latency_ms_mean"],
        row["ndcg@10"],
        color=MODEL_COLORS.get(row["model"], "#333333"),
        marker=markers.get(row["strategy"], "x"),
        s=100,
        zorder=5,
    )
    ax.annotate(
        f"{DISPLAY.get(row['model'], row['model'])}\n({row['strategy']})",
        (row["latency_ms_mean"], row["ndcg@10"]),
        textcoords="offset points",
        xytext=(6, 4),
        fontsize=7,
    )
ax.set_xlabel("Latência média por query (ms)")
ax.set_ylabel("nDCG@10")
ax.set_title("Trade-off qualidade × latência")
ax.legend(
    handles=[
        plt.Line2D([0], [0], marker=m, color="gray", linestyle="none", markersize=8, label=s.upper())
        for s, m in markers.items()
    ],
    title="Estratégia",
)
ax.grid(True, linestyle="--", alpha=0.4)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(REPORT_DIR / "eval_quality_vs_latency.png", bbox_inches="tight")
plt.show()

## 12. Tabela completa (ordenada por nDCG@10)

In [ ]:
df.select(
    "model", "strategy", *[f"ndcg@{k}" for k in K_VALUES], "latency_ms_mean", "latency_ms_p95", "n_queries"
).sort("ndcg@10", descending=True)